In [2]:
import requests, pandas as pd, hashlib
print("Mulai...")

Mulai...


In [5]:
query = """
[out:json][timeout:120];
(
  node["amenity"="cafe"](-8.8,115.1,-8.5,115.3);
  node["amenity"="restaurant"](-8.8,115.1,-8.5,115.3);
);
out;
"""
print("Lagi download data...")
r = requests.post("https://overpass-api.de/api/interpreter", data={"data": query}, timeout=120)
print("Status:", r.status_code)

# Memastikan response sukses sebelum memproses JSON
r.raise_for_status()

data = r.json()
print("Dapet elemen:", len(data['elements']))

Lagi download data...
Status: 406


HTTPError: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter

In [6]:
import requests, time

query = """
[out:json][timeout:60];
(
  node["amenity"="cafe"](-8.75,115.15,-8.60,115.25);
  node["amenity"="restaurant"](-8.75,115.15,-8.60,115.25);
);
out center 1000;
"""

servers = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter"
]

data = None
for url in servers:
    print("Coba:", url)
    try:
        r = requests.post(url, data={"data": query}, timeout=90)
        print(" Status:", r.status_code, "Panjang:", len(r.text))
        if r.status_code == 200 and len(r.text) > 100:
            data = r.json()
            print(" BERHASIL! Dapet:", len(data['elements']))
            break
    except Exception as e:
        print(" Gagal:", e)
    time.sleep(2)

if data is None:
    print("Semua server sibuk, coba lagi 2 menit ya")
else:
    print("Simpan dulu datanya...")
    import json
    with open('data_osm.json','w') as f:
        json.dump(data, f)
    print("Kesimpan!")

Coba: https://overpass-api.de/api/interpreter
 Status: 406 Panjang: 156
Coba: https://overpass.kumi.systems/api/interpreter
 Status: 429 Panjang: 89
Coba: https://overpass.private.coffee/api/interpreter
 Status: 429 Panjang: 89
Semua server sibuk, coba lagi 2 menit ya


In [1]:
import requests, time, hashlib, pandas as pd

query = """
[out:json][timeout:60];
(
  node["amenity"="cafe"](-8.75,115.15,-8.60,115.25);
  node["amenity"="restaurant"](-8.75,115.15,-8.60,115.25);
);
out center 1000;
"""

servers = [

    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
    "https://overpass.nchc.org.tw/api/interpreter",
    "https://overpass.osm.jp/api/interpreter",
]

data = None
for url in servers:
    print("Coba:", url)
    try:
        headers = {"User-Agent": "Tugas-Kuliah-BIFP-243/1.0"}
        r = requests.post(url, data={"data": query}, timeout=90, headers=headers)
        print(" Status:", r.status_code)
        if r.status_code == 200 and len(r.text) > 100:
            data = r.json()
            print(" BERHASIL! Dapet:", len(data['elements']))
            break
    except Exception as e:
        print(" Gagal:", e)
    time.sleep(2)

if data is None:
    print("Server sibuk, klik Play lagi 2 menit ya")
else:
    rows = []
    for el in data['elements']:
        tags = el.get('tags', {})
        rows.append({
            'osm_id': el['id'],
            'name': tags.get('name', ''),
            'amenity': tags.get('amenity', ''),
            'cuisine': tags.get('cuisine', ''),
            'addr_street': tags.get('addr:street', ''),
            'addr_suburb': tags.get('addr:suburb', ''),
            'phone': tags.get('phone', ''),
            'website': tags.get('website', ''),
            'opening_hours': tags.get('opening_hours', ''),
            'latitude': el.get('lat') or el.get('center', {}).get('lat'),
            'longitude': el.get('lon') or el.get('center', {}).get('lon')
        })
    df = pd.DataFrame(rows)
    df = df[df['name'] != ''].reset_index(drop=True)
    print(f"Baris: {len(df)}, Kolom: {len(df.columns)}")
    print(df.head(5).to_string())
    df.to_csv('osm_20261116.csv', index=False)
    with open('osm_20261116.csv','rb') as f:
        print("MD5:", hashlib.md5(f.read()).hexdigest())
    print("File osm_20261116.csv udah jadi!")

Coba: https://overpass-api.de/api/interpreter
 Status: 200
 BERHASIL! Dapet: 1000
Baris: 977, Kolom: 11
      osm_id              name     amenity   cuisine              addr_street addr_suburb           phone website opening_hours  latitude   longitude
0  317212141      Samudra Cafe        cafe                                                                                      -8.716141  115.168555
1  469219409  Planet Hollywood  restaurant                                                                                      -8.723245  115.182257
2  597860496              Kori  restaurant  regional                                                                            -8.717266  115.172850
3  603095835   Warung Ibu Dewi  restaurant                                                                                      -8.612942  115.247073
4  610372854   Kopi Bali House  restaurant            Jalan Bypass Ngurah Rai              +62 361 270990                       -8.704487  115.249

In [2]:
import json

provenance = {
  "nama_file": "osm_20261116.csv",
  "pengumpul": "Maryo Julio Rusbugowa Ndada",
  "nim": "2501010227",
  "email": "mariojuliondada@gmail.com",
  "sumber": "OpenStreetMap via Overpass API",
  "url_api": "https://overpass-api.de/api/interpreter",
  "tanggal_akuisisi": "2026-11-16",
  "waktu_akuisisi_wita": "2026-11-16",
  "lisensi": "Open Database License (ODbL) - https://opendatacommons.org/licenses/odbl/",
  "atribusi": "© OpenStreetMap contributors",
  "jumlah_record": 977,
  "jumlah_kolom": 11,
  "kolom": ["osm_id", "name", "amenity", "cuisine", "addr_street", "addr_suburb", "phone", "website", "opening_hours", "latitude", "longitude"],
  "area": "Denpasar, Bali, Indonesia",
  "bbox": "-8.75,115.15,-8.60,115.28",
  "filter": "amenity=cafe OR amenity=restaurant",
  "user_agent": "RTM1-2501010227 (mariojuliondada@gmail.com) - Tugas Kuliah INSTIKI",
  "md5": "a422a938f65553335084e3a436b0253f"
}

with open("provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)

print("provenance.json udah jadi!")

provenance.json udah jadi!


In [3]:
readme_text = """# RTM1 - Akuisisi Data Kafe & Resto Denpasar

**Nama:** Maryo Julio Rusbugowa Ndada
**NIM:** 2501010227
**Email:** mariojuliondada@gmail.com
**Kampus:** INSTIKI
**Mata Kuliah:** RTM1

## Deskripsi
Dataset ini berisi 977 titik kafe dan restoran di wilayah Denpasar, Bali, Indonesia.
Data diambil dari OpenStreetMap (OSM) menggunakan Overpass API.

## File dalam Project Ini
- `osm_20261116.csv` = Data utama (977 baris)
- `provenance.json` = KTP-nya data (kapan diambil, dari mana, MD5)
- `README.md` = File penjelasan ini

## Cara Pengambilan Data
1. Sumber: OpenStreetMap via Overpass API
2. Area: Denpasar, Bali (bbox -8.75,115.15,-8.60,115.28)
3. Filter: amenity=cafe OR amenity=restaurant
4. Tanggal: 2026-11-16

## Lisensi
Data ini berlisensi Open Database License (ODbL).
Atribusi: © OpenStreetMap contributors
Link: https://opendatacommons.org/licenses/odbl/

## Kolom Data
osm_id, name, amenity, cuisine, addr_street, addr_suburb, phone, website, opening_hours, latitude, longitude
"""

with open("README.md", "w") as f:
    f.write(readme_text)

print("README.md udah jadi!")

README.md udah jadi!


In [5]:
# Tugas 1 - Akuisisi OSM
# Nama: Maryo Julio Rusbugowa Ndada
# NIM: 2501010227
# Email: mariojuliondada@gmail.com

!pip install osmnx -q

import osmnx as ox
import pandas as pd
import time

# Coba server cadangan kalau server utama penuh
endpoints = [
    "https://overpass.private.coffee/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass-api.de/api/interpreter"
]

gdf = None
for ep in endpoints:
    try:
        print(f"Nyoba server: {ep}")
        ox.settings.overpass_endpoint = ep
        tags = {"amenity": ["cafe", "restaurant"]}
        gdf = ox.features_from_place("Denpasar, Bali, Indonesia", tags=tags)
        print(f"BERHASIL! Dapet {len(gdf)} baris mentah")
        break
    except Exception as e:
        print(f"Server ini penuh, pindah ke berikutnya...")
        time.sleep(5)

# Bersihin
gdf = gdf.reset_index()
df = pd.DataFrame()
df["osm_id"] = gdf["id"] if "id" in gdf.columns else gdf.index
df["name"] = gdf["name"] if "name" in gdf.columns else ""
df["amenity"] = gdf["amenity"] if "amenity" in gdf.columns else ""
df["cuisine"] = gdf["cuisine"] if "cuisine" in gdf.columns else ""
df["addr_full"] = gdf["addr:full"] if "addr:full" in gdf.columns else ""
df["addr_street"] = gdf["addr:street"] if "addr:street" in gdf.columns else ""
df["phone"] = gdf["phone"] if "phone" in gdf.columns else ""
df["website"] = gdf["website"] if "website" in gdf.columns else ""
df["opening_hours"] = gdf["opening_hours"] if "opening_hours" in gdf.columns else ""
df["lon"] = gdf.geometry.centroid.x
df["lat"] = gdf.geometry.centroid.y

df = df.dropna(subset=["name"])
df = df[df["name"] != ""]
df = df.drop_duplicates(subset=["name", "lat", "lon"])

df.to_csv("osm_20261116.csv", index=False)
print(f"BERES! Tersimpan {len(df)} baris ke osm_20261116.csv")
df.head()

Nyoba server: https://overpass.private.coffee/api/interpreter
BERHASIL! Dapet 793 baris mentah
BERES! Tersimpan 771 baris ke osm_20261116.csv


/tmp/ipykernel_1096/136493562.py:44: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df["lon"] = gdf.geometry.centroid.x
/tmp/ipykernel_1096/136493562.py:45: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df["lat"] = gdf.geometry.centroid.y


,osm_id,name,amenity,cuisine,addr_full,addr_street,phone,website,opening_hours,lon,lat
0,603104018,Splash Bakery,restaurant,NaN,,NaN,NaN,NaN,NaN,115.258695,-8.679811
1,610372854,Kopi Bali House,restaurant,NaN,,Jalan Bypass Ngurah Rai,+62 361 270990,NaN,NaN,115.249949,-8.704487
2,610374517,Randy's,restaurant,NaN,,NaN,NaN,NaN,NaN,115.262193,-8.685404
3,610375066,The Porch Cafe,restaurant,NaN,,NaN,NaN,NaN,NaN,115.263190,-8.690896
4,610375768,Smorga's,restaurant,NaN,,NaN,NaN,NaN,NaN,115.263789,-8.693603


In [6]:
import pandas as pd

# Data Ethics Checklist - Tugas 1
# Nama: Maryo Julio Rusbugowa Ndada
# NIM: 2501010227

data = [
    ["1", "Sumber data legal & terbuka", "Ya", "Data dari OpenStreetMap (OSM), sumber terbuka, lisensi ODbL"],
    ["2", "Tidak ada data pribadi", "Ya", "Hanya nama tempat, alamat umum, koordinat. Tidak ada nama orang, NIK, no HP pribadi"],
    ["3", "Tidak ada scraping akun pribadi / login", "Ya", "Menggunakan API Overpass OSM, tanpa login, tanpa scraping Instagram/WA"],
    ["4", "Atribusi sumber dicantumkan", "Ya", "Sumber: OpenStreetMap contributors, disebutkan di README dan provenance.json"],
    ["5", "Lisensi dihormati", "Ya", "Lisensi ODbL - data boleh dipakai, dibagikan, dengan atribusi"],
    ["6", "Tidak ada data sensitif / anak / kesehatan", "Ya", "Data hanya cafe & restaurant di Denpasar, data publik"],
    ["7", "Data tidak dimanipulasi", "Ya", "Data apa adanya dari OSM, hanya dibersihkan duplikat & nama kosong"],
    ["8", "Privasi lokasi dihormati", "Ya", "Lokasi adalah tempat usaha publik, bukan rumah pribadi"],
    ["9", "Penggunaan untuk akademik", "Ya", "Hanya untuk Tugas 1 RTM1, tidak untuk komersial"],
    ["10", "File provenance dilampirkan", "Ya", "provenance.json menjelaskan asal, tanggal, dan cara ambil data"],
]

df = pd.DataFrame(data, columns=["no", "check", "status", "catatan"])
df.to_csv("data_ethics_checklist.csv", index=False)
print("BERES! data_ethics_checklist.csv kebikin")
df

BERES! data_ethics_checklist.csv kebikin


,no,check,status,catatan
0,1,Sumber data legal & terbuka,Ya,"Data dari OpenStreetMap (OSM), sumber terbuka,..."
1,2,Tidak ada data pribadi,Ya,"Hanya nama tempat, alamat umum, koordinat. Tid..."
2,3,Tidak ada scraping akun pribadi / login,Ya,"Menggunakan API Overpass OSM, tanpa login, tan..."
3,4,Atribusi sumber dicantumkan,Ya,"Sumber: OpenStreetMap contributors, disebutkan..."
4,5,Lisensi dihormati,Ya,"Lisensi ODbL - data boleh dipakai, dibagikan, ..."
5,6,Tidak ada data sensitif / anak / kesehatan,Ya,"Data hanya cafe & restaurant di Denpasar, data..."
6,7,Data tidak dimanipulasi,Ya,"Data apa adanya dari OSM, hanya dibersihkan du..."
7,8,Privasi lokasi dihormati,Ya,"Lokasi adalah tempat usaha publik, bukan rumah..."
8,9,Penggunaan untuk akademik,Ya,"Hanya untuk Tugas 1 RTM1, tidak untuk komersial"
9,10,File provenance dilampirkan,Ya,"provenance.json menjelaskan asal, tanggal, dan..."
